In [ ]:
import pandas as pd
import requests
import io
import os
import time
from urllib.parse import urlparse, parse_qs

In [ ]:
# KONFIGURATION

INPUT_CSV = "api_requests.csv"       # Dateiname deiner angehängten CSV anpassen
OUTPUT_DIR = "geosphere_downloads"   # Zielordner für die einzelnen CSV-Dateien
ENDPOINT = "https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-1d"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# DATEI LADEN & GEWÜNSCHTE DATEINAMEN EXTRAHIEREN
df = pd.read_csv(INPUT_CSV)
df['date_obj'] = pd.to_datetime(df['before_date'])

# Extrahiert den exakten 'filename' Parameter aus jeder API-URL
def extract_filename(url):
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    return qs.get('filename', [None])[0]

df['target_filename'] = df['api_request'].apply(extract_filename)

# Duplikate entfernen (falls zwei Brände denselben Tag & dieselbe Station abfragen)
unique_tasks = df[['station_id', 'date_obj', 'before_date', 'target_filename']].drop_duplicates()
unique_tasks = unique_tasks.sort_values(by=['station_id', 'date_obj']).reset_index(drop=True)

print(f"[+] {len(df)} Zeilen eingelesen -> {len(unique_tasks)} einzigartige (Station + Tag)-Kombinationen.")

In [ ]:
# ZUSAMMENHÄNGENDE DATUMSBLÖCKE PRO STATION BÜNDELN
# Erkennt automatisch Lücken von mehr als 1 Tag und bildet zusammenhängende Blöcke
unique_tasks['date_diff'] = unique_tasks.groupby('station_id')['date_obj'].diff().dt.days
unique_tasks['block_id'] = (unique_tasks['date_diff'].isna() | (unique_tasks['date_diff'] > 1)).cumsum()

grouped_blocks = unique_tasks.groupby(['station_id', 'block_id']).agg(
    start_date=('date_obj', 'min'),
    end_date=('date_obj', 'max'),
    days_count=('date_obj', 'count')
).reset_index()

print(f"[+] Durch Bündelung auf nur {len(grouped_blocks)} API-Requests reduziert!\n")


# API ABFRAGEN & EINZELDATEIEN LOKAL SPEICHERN

target_parameters = [
    "tl_ii", "tlmax", "vvbft_ii", "rfb_ii", "rf_ii", "rf_mittel",
    "rr_iii", "sh", "p_mittel", "rfb_mittel", "tl_mittel", "vv_mittel"
]

# Lookup-Dictionary für die schnellen Dateinamen: (station_id, 'YYYY-MM-DD') -> 'filename.csv'
filename_lookup = {
    (row['station_id'], row['before_date']): row['target_filename']
    for _, row in unique_tasks.iterrows()
}

saved_files_count = 0

for idx, block in grouped_blocks.iterrows():
    st_id = block['station_id']
    start_str = block['start_date'].strftime('%Y-%m-%dT00:00:00')
    end_str = block['end_date'].strftime('%Y-%m-%dT00:00:00')
    
    params = {
        "parameters": target_parameters,
        "start": start_str,
        "end": end_str,
        "station_ids": str(st_id),
        "output_format": "csv"
    }
    
    print(f"Rufe Station {st_id} ab ({block['start_date'].strftime('%Y-%m-%d')} bis {block['end_date'].strftime('%Y-%m-%d')})...")
    
    try:
        response = requests.get(ENDPOINT, params=params)
        
        
        if response.status_code == 200:
            raw_text = response.text.strip()
            
            if raw_text.startswith("<"):
                print(f"[!] Station {st_id}: HTML-Antwort statt CSV erhalten.")
                continue
                
            block_df = pd.read_csv(io.StringIO(raw_text))
            
            if block_df.empty:
                print(f"[-] Station {st_id}: Keine Messdaten in diesem Zeitraum vorhanden.")
                continue
                
            # Datum aus der 'time'-Spalte extrahieren (Format: YYYY-MM-DD)
            block_df['date_str'] = pd.to_datetime(block_df['time']).dt.strftime('%Y-%m-%d')
            
            # Jede einzelne Tageszeile als eigene CSV-Datei speichern
            for _, day_row in block_df.iterrows():
                d_str = day_row['date_str']
                fname = filename_lookup.get((st_id, d_str))
                
                # Fallback-Dateiname, falls in der URL kein filename-Parameter stand
                if not fname:
                    fname = f"geosphere.at_{d_str}_{st_id}_dataset.csv"
                    
                # Hilfsspalte 'date_str' vor dem Speichern wieder entfernen
                single_day_df = pd.DataFrame([day_row.drop(labels=['date_str'])])
                file_path = os.path.join(OUTPUT_DIR, fname)
                
                single_day_df.to_csv(file_path, index=False)
                saved_files_count += 1
                
            print(f"  -> {len(block_df)} Einzeldateien erfolgreich gespeichert.")
            
        elif response.status_code == 429:
            print("[!] Rate Limit erreicht! Warte 60 Sekunden...")
            time.sleep(60)
        else:
            print(f"[!] HTTP Fehler {response.status_code} bei Station {st_id}")
            
    except Exception as e:
        print(f"[!] Fehler bei Station {st_id}: {e}")
        
    # Schonende Pause zwischen den gebündelten Requests
    time.sleep(0.5)

print(f"\nFertig! Insgesamt {saved_files_count} Einzeldateien im Ordner '{OUTPUT_DIR}/' gespeichert.")


[+] 364 Zeilen eingelesen -> 364 einzigartige (Station + Tag)-Kombinationen.
[+] Durch Bündelung auf nur 26 API-Requests reduziert!

Rufe Station 46 ab (2015-07-08 bis 2015-07-21)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 69 ab (2015-11-23 bis 2015-12-06)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 72 ab (2015-03-04 bis 2015-03-17)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 93 ab (2012-11-27 bis 2012-12-10)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 108 ab (2012-04-15 bis 2012-04-28)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 108 ab (2013-07-24 bis 2013-08-06)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 112 ab (2013-07-18 bis 2013-07-31)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 112 ab (2015-07-11 bis 2015-07-24)...
  -> 14 Einzeldateien erfolgreich gespeichert.
Rufe Station 120 ab (2013-07-12 bis 2013-07-25)...
  -> 14 Einzeldateien erfolgreich ge